# Benchmark: Simple vs Expanded ELBO Modes

This notebook compares the two ELBO computation modes in PNMF:

## `mode='simple'`
Uses `torch.distributions.Poisson.log_prob()` directly with Monte Carlo estimation.

## `mode='expanded'` (default)
Uses a hybrid approach:
- First term: Monte Carlo estimation for `Y * E[log(rate)]`
- Second term: Analytic computation for `E[exp(F)]` using Gaussian moment-generating function
- Third term: Poisson normalization constant `log(Y!)`

### Key Differences

| Aspect | Simple | Expanded |
|--------|--------|----------|
| Variance | Higher (full MC) | Lower (hybrid) |
| Computation | Direct PyTorch call | Custom implementation |
| Convergence | May be slower | Often faster |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

from PNMF import PNMF
from PNMF.priors import GaussianPrior
from PNMF.models import PoissonFactorization

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Generate Synthetic Data

Create data that approximately follows the PNMF model. The internal representation uses:
- `X (D, N) ≈ W (D, L) @ exp(F) (L, N)` where D=features, N=samples, L=components
- For sklearn API (n_samples, n_features): `X (N, D) ≈ exp(F).T (N, L) @ W.T (L, D)`

In [ ]:
def generate_synthetic_data(n_samples=200, n_features=100, n_components=5, random_state=42):
    """
    Generate synthetic non-negative data for benchmarking.
    """
    rng = np.random.RandomState(random_state)

    # Generate true W (positive)
    W_true = rng.exponential(scale=1.0, size=(n_features, n_components))

    # Generate true F (Gaussian latent factors)
    F_true = rng.randn(n_components, n_samples)

    # Compute X
    X = W_true @ np.exp(F_true)

    # Add some noise
    X += rng.exponential(scale=0.1, size=X.shape)

    # Ensure non-negative
    X = np.maximum(X, 0)

    return X

# Generate data
X = generate_synthetic_data(n_samples=200, n_features=100, n_components=5, random_state=42)
print(f"Data shape: {X.shape}")
print(f"Data range: [{X.min():.2f}, {X.max():.2f}]")
print(f"Data mean: {X.mean():.2f}")

## 2. Define Training Functions

Create functions to train both modes while tracking ELBO convergence.

In [ ]:
def train_with_elbo_tracking(X, mode='expanded', n_components=5, max_iter=200, random_state=42):
    """
    Train PNMF with specified mode and track ELBO convergence.
    
    Returns:
        results: Dictionary with ELBO history and final metrics
    """
    # Convert to torch tensor
    X_torch = torch.from_numpy(X.T.astype(np.float32))

    # Initialize prior and model
    torch.manual_seed(random_state)
    prior = GaussianPrior(y=X_torch, L=n_components)
    pf_model = PoissonFactorization(
        prior=prior, 
        y=X_torch, 
        L=n_components, 
        loadings_mode='projected', 
        mode=mode
    )

    # Setup optimizer
    params = list(pf_model.W.parameters()) + list(prior.parameters())
    optimizer = torch.optim.Adam(params, lr=0.01)

    # Training loop with ELBO tracking
    elbo_history = []
    prev_elbo = float('-inf')

    for iteration in tqdm(range(max_iter), desc=f"{mode} mode"):
        optimizer.zero_grad()
        rate, qF, pF = pf_model.forward(E=3)

        # Compute ELBO based on mode
        if mode == 'simple':
            # Simple mode: full Monte Carlo estimation
            E_samples = rate.shape[0]
            eps = 1e-8
            rate_clamped = rate.clamp(min=eps)
            X_expanded = X_torch.unsqueeze(0).expand(E_samples, -1, -1)
            # All terms via Monte Carlo: X * log(rate) - rate - log(X!)
            log_lik_mc = (X_expanded * torch.log(rate_clamped) - rate_clamped - torch.lgamma(X_expanded + 1))
            log_lik = log_lik_mc.sum() / E_samples
        else:
            # Expanded mode: hybrid MC + analytic
            E_samples = rate.shape[0]
            eps = 1e-8
            rate_clamped = rate.clamp(min=eps)
            X_expanded = X_torch.unsqueeze(0).expand(E_samples, -1, -1)
            term1_mc = (X_expanded * torch.log(rate_clamped)).sum() / E_samples
            mu = qF.mean
            sigma = qF.scale
            exp_expectation = torch.exp(mu + 0.5 * sigma ** 2)
            W = pf_model.W.data
            term2_analytic = torch.matmul(W, exp_expectation).sum()
            log_lik = term1_mc - term2_analytic - torch.lgamma(X_torch + 1).sum()

        kl = torch.distributions.kl_divergence(qF, pF).sum()
        loss = kl - log_lik

        loss.backward()
        optimizer.step()

        # Project parameters
        pf_model.project_parameters()

        elbo_value = -loss.item()
        elbo_history.append(elbo_value)

        # Check convergence
        if abs(elbo_value - prev_elbo) < 1e-4:
            break
        prev_elbo = elbo_value

    # Compute reconstruction error
    with torch.no_grad():
        rate_final, _, _ = pf_model.forward(E=10)
        rate_mean = rate_final.mean(dim=0).t().numpy()  # (N, D)
        reconstruction_error = np.linalg.norm(X - rate_mean, 'fro') / np.linalg.norm(X, 'fro')

    return {
        'mode': mode,
        'n_iterations': len(elbo_history),
        'final_elbo': elbo_history[-1],
        'elbo_history': elbo_history,
        'reconstruction_error': reconstruction_error,
        'converged': len(elbo_history) < max_iter
    }


## 3. Run Benchmarks

Train both models with identical initialization and data.

In [ ]:
# Run simple mode benchmark
print("Running simple mode (torch.Poisson)...")
results_simple = train_with_elbo_tracking(
    X, 
    mode='simple', 
    n_components=5, 
    max_iter=200, 
    random_state=42
)
print(f"  Completed in {results_simple['n_iterations']} iterations")
print(f"  Final ELBO: {results_simple['final_elbo']:.4f}")
print()

In [ ]:
# Run expanded mode benchmark
print("Running expanded mode (hybrid MC + analytic)...")
results_expanded = train_with_elbo_tracking(
    X, 
    mode='expanded', 
    n_components=5, 
    max_iter=200, 
    random_state=42
)
print(f"  Completed in {results_expanded['n_iterations']} iterations")
print(f"  Final ELBO: {results_expanded['final_elbo']:.4f}")

## 4. Compare Results

Display a summary table comparing the two modes.

In [ ]:
# Print summary
print("=" * 70)
print("PNMF Benchmark: Simple vs Expanded ELBO Modes")
print("=" * 70)
print()

print(f"{'Metric':<30} {'Simple':<20} {'Expanded':<20}")
print("-" * 70)

# Iterations to convergence
print(f"{'Iterations to convergence':<30} "
      f"{results_simple['n_iterations']:<20} "
      f"{results_expanded['n_iterations']:<20}")

# Final ELBO
print(f"{'Final ELBO':<30} "
      f"{results_simple['final_elbo']:<20.4f} "
      f"{results_expanded['final_elbo']:<20.4f}")

# ELBO difference
elbo_diff = abs(results_simple['final_elbo'] - results_expanded['final_elbo'])
print(f"{'ELBO difference':<30} {'':<20} {elbo_diff:<20.6f}")

# Reconstruction error
print(f"{'Relative reconstruction error':<30} "
      f"{results_simple['reconstruction_error']:<20.6f} "
      f"{results_expanded['reconstruction_error']:<20.6f}")

print()

# Winner analysis
if results_simple['n_iterations'] < results_expanded['n_iterations']:
    faster = "Simple"
    speedup = results_expanded['n_iterations'] / results_simple['n_iterations']
else:
    faster = "Expanded"
    speedup = results_simple['n_iterations'] / results_expanded['n_iterations']

print(f"Convergence winner: {faster} ({speedup:.2f}x faster)")
print()

# ELBO comparison
if results_simple['final_elbo'] > results_expanded['final_elbo']:
    print(f"Final ELBO winner: Simple (higher by {elbo_diff:.6f})")
elif results_expanded['final_elbo'] > results_simple['final_elbo']:
    print(f"Final ELBO winner: Expanded (higher by {elbo_diff:.6f})")
else:
    print("Final ELBO: Tie")

print("=" * 70)

## 5. Visualize Convergence

Plot the ELBO convergence curves for both modes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: ELBO convergence
ax1 = axes[0]
ax1.plot(results_simple['elbo_history'], label='Simple (torch.Poisson)', linewidth=2)
ax1.plot(results_expanded['elbo_history'], label='Expanded (hybrid)', linewidth=2)
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('ELBO', fontsize=12)
ax1.set_title('ELBO Convergence Comparison', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: ELBO difference (relative to final)
ax2 = axes[1]
final_simple = results_simple['elbo_history'][-1]
final_expanded = results_expanded['elbo_history'][-1]

# Plot distance to convergence
diff_simple = [abs(x - final_simple) for x in results_simple['elbo_history']]
diff_expanded = [abs(x - final_expanded) for x in results_expanded['elbo_history']]

ax2.semilogy(diff_simple, label='Simple (torch.Poisson)', linewidth=2)
ax2.semilogy(diff_expanded, label='Expanded (hybrid)', linewidth=2)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('|ELBO - Final|', fontsize=12)
ax2.set_title('Distance to Convergence (log scale)', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Key Takeaways

Based on the results above:

1. **Convergence Speed**: Which mode converged faster?
2. **Final ELBO**: Did both modes reach similar ELBO values?
3. **Stability**: Which mode had more stable convergence?

The `expanded` mode typically has lower variance due to the analytic computation of the second term, which can lead to faster and more stable convergence.